In [1]:
# 使用A100进行训练
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("./nlpcc_2017/")
ds

Dataset({
    features: ['title', 'content'],
    num_rows: 5000
})

In [4]:
ds = ds.train_test_split(500, seed=42)
ds

DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['title', 'content'],
        num_rows: 500
    })
})

In [5]:
ds["train"][0]

{'title': '欧盟宣布启动 “欧盟地中海海军”计划,打击地中海人口走私贩运,将与非洲国家、国际移民组织共同应对。',
 'content': '新华网布鲁塞尔6月22日电(记者周珺孙奕)欧盟成员国外长会22日在卢森堡举行,会议宣布启动在地中海打击人口走私贩运的“欧盟地中海海军”行动计划。欧盟外交和安全政策高级代表莫盖里尼说:“欧盟从未如此重视移民问题,我们的目标是打击从移民的苦难中获益的商业模式。但这只是欧盟一个更广泛战略中的一部分。”她表示,欧盟将与非洲国家尤其是萨赫勒地区国家合作,并与国际移民组织和联合国难民署共同应对移民问题。“欧盟地中海海军”行动计划分为三个阶段,第一阶段重点监测和评估地中海人口走私和贩运网络;第二阶段将展开行动搜索和检查可疑船只;第三阶段将逮捕人口走私和贩运者,并对船只及相关资产进行处置。当天会后发布的公报指出,目前只是宣布启动第一阶段行动。考虑到联合国的授权和有关沿海国家的意见,欧盟将评估何时展开第二阶段行动。欧盟5月18日决定采取军事行动打击地中海人口走私活动。该行动的总部设在意大利罗马,欧盟初期为这一行动投入1182万欧元。联合国秘书长潘基文5月27日在布鲁塞尔表示,使用军事手段打击偷渡效果有限,需通盘考虑偷渡者的来源地、中转地、目的地等因素,以全面应对偷渡问题。'}

In [6]:
# 对于高版本的Transformers加载会报错，需要修改源码
# 文件地址 ~/.cache\huggingface\modules\transformers_modules\THUDM\glm-large-chinese\230f54e413fab4bc8f29bd3508aab301d757ef3e\tokenization_glm.py
# 231行 super().__init__(**kwargs) 移动至 235行，放至self.sp_model.Load(vocab_file)的后面一行
tokenizer = AutoTokenizer.from_pretrained("THUDM/glm-large-chinese", trust_remote_code=True)
tokenizer

GLMChineseTokenizer(name_or_path='THUDM/glm-large-chinese', vocab_size=50000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='left', special_tokens={'eos_token': '<|endoftext|>', 'unk_token': '[UNK]', 'pad_token': '<|endoftext|>', 'cls_token': '[CLS]', 'mask_token': '[MASK]', 'additional_special_tokens': ['<|startofpiece|>', '<|endofpiece|>', '[gMASK]', '[sMASK]']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50001: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50002: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50003: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50004: AddedToken("[UNUSED1]", rstrip=False, lstrip=False, single_word=Fal

In [7]:
def process_func(exmaples):
    contents = ["摘要生成: \n" + e + tokenizer.mask_token for e in exmaples["content"]]
    inputs = tokenizer(contents, max_length=384, truncation=True, padding="max_length", return_tensors="pt")
    inputs = tokenizer.build_inputs_for_generation(inputs, targets=exmaples['title'], padding=True, max_gen_length=64)
    return inputs

In [8]:
tokenized_ds = ds.map(process_func, batched=True, remove_columns=ds["train"].column_names)
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'position_ids', 'attention_mask', 'labels'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['input_ids', 'position_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})

In [9]:
tokenizer.decode(tokenized_ds["train"][0]["input_ids"])

'[CLS] 摘要生成: 新华网布鲁塞尔6月22日电(记者周珺孙奕)欧盟成员国外长会22日在卢森堡举行,会议宣布启动在地中海打击人口走私贩运的“欧盟地中海海军”行动计划。欧盟外交和安全政策高级代表莫盖里尼说:“欧盟从未如此重视移民问题,我们的目标是打击从移民的苦难中获益的商业模式。但这只是欧盟一个更广泛战略中的一部分。”她表示,欧盟将与非洲国家尤其是萨赫勒地区国家合作,并与国际移民组织和联合国难民署共同应对移民问题。“欧盟地中海海军”行动计划分为三个阶段,第一阶段重点监测和评估地中海人口走私和贩运网络;第二阶段将展开行动搜索和检查可疑船只;第三阶段将逮捕人口走私和贩运者,并对船只及相关资产进行处置。当天会后发布的公报指出,目前只是宣布启动第一阶段行动。考虑到联合国的授权和有关沿海国家的意见,欧盟将评估何时展开第二阶段行动。欧盟5月18日决定采取军事行动打击地中海人口走私活动。该行动的总部设在意大利罗马,欧盟初期为这一行动投入1182万欧元。联合国秘书长潘基文5月27日在布鲁塞尔表示,使用军事手段打击偷渡效果有限,需通盘考虑偷渡者的来源地、中转地、目的地等因素,以全面应对偷渡问题。[MASK]<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|e

In [10]:
print(tokenized_ds["train"][0]["labels"])

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

In [11]:
print(tokenized_ds["train"][0]["position_ids"])

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221

In [12]:
model = AutoModelForSeq2SeqLM.from_pretrained("THUDM/glm-large-chinese", trust_remote_code=True)

In [13]:
args = Seq2SeqTrainingArguments(
    output_dir="./summary_glm",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    logging_steps=20,
    num_train_epochs=2
)

In [14]:
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
)  

/tmp/ipykernel_1269/2961098060.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-12-03 14:30:34,423] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-12-03 14:30:35,971] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [15]:
trainer.train()

Step,Training Loss
20,7.650800
40,6.963900
60,6.551900
80,6.517800
100,6.328900
120,6.458600
140,6.077700
160,4.743400
180,4.632800
200,4.611600


TrainOutput(global_step=282, training_loss=5.629070261691479, metrics={'train_runtime': 1593.9483, 'train_samples_per_second': 5.646, 'train_steps_per_second': 0.177, 'total_flos': 8553337454592000.0, 'train_loss': 5.629070261691479, 'epoch': 2.0})

In [16]:
input_text = ds["test"][-1]["content"]
inputs = tokenizer("摘要生成: \n" + input_text + tokenizer.mask_token, return_tensors="pt")
inputs = tokenizer.build_inputs_for_generation(inputs, max_gen_length=64)
inputs = inputs.to("cuda")
output = model.generate(**inputs, max_new_tokens=64, eos_token_id=tokenizer.eop_token_id, do_sample=True)
tokenizer.decode(output[0].tolist())

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'[CLS] 摘要生成: 仅仅因为恋人的一个意外电话,被告人管某便在加油站旁将汽油泼到他人身上,扬言纵火。近日,山东省临沂市兰山区人民法院以管某犯以危险方法危害公共安全罪,判处有期徒刑三年。据了解,管某与周某系恋人关系,2014年11月4日19时许,周某到管某住处时,见管某正与其娘家的哥哥尤某打电话,并有意避讳周某,周某便怀疑尤某勾引管某,遂打电话与尤某争吵并相约在某中国石化加油站西边红绿灯处见面。随后,周某提一油漆桶到该加油站购买6.74升97#汽油,等候尤某。当晚21时许,当尤某到达该加油站内时,周某将汽油泼到尤某的身上,并手持打火机扬言要将其烧死,被正在巡逻的派出所民警发现并及时制止。在案件审理过程中,周某当庭翻供,与其辩护人辩称其在现场所使用的打火机是坏的不能点燃。法院认为,该打火机是否能点燃,不影响该犯罪构成。加油站本身是特殊地点,应当谨慎行为是普通常识,而周某在这种特殊地点实施了泼洒他人汽油的行为,已对周围不特定多数人的人身安全和公私财产构成威胁,其行为已构成以危险方法危害公共安全罪,应予惩处。于是判处有期徒刑三年。(胡雪莹张琳)[MASK]<|endoftext|> <|startofpiece|> 临沂一男子在加油站旁向路人泼汽油,扬言扬火烧死他;因与恋人通电话被怀疑,遂买汽油约见恋人。 <|endofpiece|>'

In [25]:
ds_test = ds["test"].select(range(10))
ds_test

Dataset({
    features: ['title', 'content'],
    num_rows: 10
})

In [26]:
import torch

model = model.eval()

def predict_test():
    predict = []
    with torch.inference_mode():
        for d in ds_test:
            inputs = tokenizer("摘要生成: \n" + d["content"] + tokenizer.mask_token, return_tensors="pt")
            inputs = tokenizer.build_inputs_for_generation(inputs, max_gen_length=64)
            inputs = inputs.to("cuda")
            output = model.generate(**inputs, max_new_tokens=64, eos_token_id=tokenizer.eop_token_id, do_sample=True)
            predict.append(tokenizer.decode(output[0].tolist()).split("<|startofpiece|>")[1].replace("<|endofpiece|>", "").strip())
            print("curID:", len(predict))
    return predict

In [27]:
result = predict_test()

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.


curID: 1


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.


curID: 2


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.


...

curID: 7


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.


curID: 8


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50007 for open-end generation.


curID: 9
curID: 10


In [28]:
result

['台媒称IS公布1400名人名单,大部分是美国人,包括美国总统及其家属;IS旗下黑客声称已将有关资料发给“圣战者”,准备将其列为攻击目标。',
 '宿松县2名副局长因嫖娼被双开,被指多次吸毒被行政拘留13天,分管食药工商的乡镇主要负责人被通报批评。',
 '黑龙江“伪基站”使用“骚扰”成气候 每天影响手机用户190万',
 '明天后五,北京将重启限行,周一、四、五、日限行轮换,本轮限行与北京保持一致。',
 '哈尔滨男子有外遇致妻离子散,母亲与妻子找上门,两人因琐事在宾馆厮打5分钟,辅警赶到后劝和劝解离开现场',
 '苏州吴江一初中副校长泄考题被停职,称纪委已介入调查,校方已推出中考改革方案挽回影响。',
 '承德广电2厅级干部被双开:承德广播电视台0752转播台原台长高登银、方向云2人严重违纪被双开。',
 '“中国骄傲”李佩斯开通微博,当天发了一张自己的照片,上面写着“大家好,我是李佩斯”;网友纷纷留言表示“这中文是用尺子量着写的吗?”',
 '外媒称“IS”在叙利亚和伊拉克攫取小麦已成其势力范围的战略轴心;据悉“IS”通过非法“过境土耳其”向“基地”组织等成员国出售小麦。',
 '天津市发布大风黄色预警:预计今日夜间到明天白天渤海海面,将有西北风9级,阵风10-11级。请有关单位和人员作好防范准备。...']

In [29]:
from rouge_chinese import Rouge

rouge = Rouge()

docode_preds = [" ".join(p) for p in result]
decode_labels = [" ".join(l) for l in ds_test["title"]]
scores = rouge.get_scores(docode_preds, decode_labels, avg=True)
{
    "rouge-1": scores["rouge-1"]["f"],
    "rouge-2": scores["rouge-2"]["f"],
    "rouge-l": scores["rouge-l"]["f"],
}

{'rouge-1': 0.518773100501855,
 'rouge-2': 0.30837470611814627,
 'rouge-l': 0.42711833820905537}